# Fleet Observatory · check monthly claim activity

**The 30-day collection succeeded. This notebook investigates its incomplete payout evidence.**

Your completed 19-group check and saved analysis notebook have been inspected. All 19 checks succeeded and the executed notebook has no error outputs. This follow-up uses the **same 6 September–5 October 2026 UTC month, eight services, original settlement cutoff and exact sampled session IDs** as the website. It does not download another month or change the website's calculations.

It contains all its Python code, tests and required diagnostic inputs. It uses Python's standard library only: **no packages, API key, Node.js, GitHub or Vercel are required.** The full 9 MB snapshot is not repeated; this notebook carries the exact 4,080 session IDs and source checksum needed for these read-only count checks.

**First:** download this notebook into the same folder as your other notebooks. Open it in Jupyter, leave `MODE = "offline"`, select **Run → Run All Cells**, and save with **Ctrl+S**. Expect **16 tests passed**; no live requests occur.

**Then:** change `MODE` to `"verify"`, run all cells again, and save. A fresh run makes **39 sequential HTTP requests** with progress after each request and group. Each request has a 45-second timeout and no automatic retries. This is a small diagnostic, although a slow API can still take several minutes. If interrupted, rerunning resumes the saved groups. A completed result is reused with zero new requests.

**Return:** `monthly-claims-check.json` and this saved notebook. If it fails, return `monthly-claims-error.json` and the notebook. The last cell prints their full paths.


## 1 · What your 19-group result establishes

| Result | Interpretation |
|---|---|
| 15 groups returned historical claim counts | These Application/service pairs have indexed settled-claim history through the saved cutoff. |
| 4 groups returned a count of zero | No indexed settled claims matched the history query through that cutoff. This does not prove zero traffic or income. |
| All 72 example claims predate 6 September | Consistent with old activity, but the examples are not a reliable last-activity list. |
| Indexer processed height 958,539; saved cutoff 955,836 | The reported indexer head was beyond the saved cutoff. This alone does not prove every event was indexed correctly. |
| 19 groups account for 2,280 of 2,428 unknown samples | 93.9% of the missing sampled payout evidence is concentrated in these groups. |

An **Application/service group** means a customer account using one service. Fleet groups on the website describe Suppliers grouped by recorded domains or owner addresses. These are different partitions of the same session evidence.

**Correction to the first diagnostic:** its `latestFiveClaims` field used `orderBy: ID_DESC`. Those event IDs contain text, so that ordering does not guarantee latest block or date. The historical counts remain useful, but we cannot infer last activity from those five examples. This notebook uses explicit numeric height filters and never calls example rows “latest.”

Reference: [SubQuery's documentation on ID types and text ordering](https://subquery.network/doc/indexer/build/graphql/reference.html#ids). The API's actual supported filter types are checked before live count requests.

The completed queries provide no evidence of you being kicked off the API during that check. Missing claims in a successfully returned result are a separate issue from a rejected or timed-out request.


In [ ]:
from pathlib import Path
import hashlib, importlib, json, subprocess, sys, time

MODE = "offline"        # First run: "offline". Second run: "verify".
NOTEBOOK_FILE = None     # Auto-find this saved file, or enter its full local path.
WORKDIR = Path.cwd() / "pokt-monthly-claims-work"
CODEDIR = WORKDIR / "source"
OUTPUT = WORKDIR / "results"
REQUEST_TIMEOUT_SECONDS = 45
for folder in [CODEDIR, OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)
if MODE not in {"offline", "verify"}:
    raise ValueError('MODE must be "offline" or "verify"')
print("Mode:", MODE)
print("Working folder:", WORKDIR.resolve())
print("Keep this notebook saved on disk. There is no full-month collection in this notebook.")


## 2 · Load the same month and your completed history check

The small embedded manifest is stored in the saved notebook's metadata so Jupyter does not have to edit a large data cell. It includes the precise original session IDs, per-group counts and the uploaded 19-group result. It is checksum-checked before use. No other input file is needed.

If Jupyter opens from a different directory and the next cell cannot find this notebook, set `NOTEBOOK_FILE` above to its complete path, for example `Path(r"C:\Users\mwil8\Documents\1 --- pokt\pokt-notebook\pokt_monthly_claims_check.ipynb")`. Use the path where you actually saved it. Run all cells again.


In [ ]:
candidates = [Path(NOTEBOOK_FILE)] if NOTEBOOK_FILE else sorted(
    Path.cwd().glob("pokt_monthly_claims_check*.ipynb"),
    key=lambda p: p.stat().st_mtime, reverse=True)
embedded = None
for candidate in candidates:
    saved = json.loads(candidate.read_text(encoding="utf-8"))
    item = saved.get("metadata", {}).get("pokt_monthly_claims_input")
    if item and hashlib.sha256(json.dumps(item, sort_keys=True, separators=(",", ":")).encode()).hexdigest() == '538ba3c5c61fe2a4e41ea919b23dd3b7bde423b4dc3be9a292f9b5daddb00250':
        embedded = item
        print("Loaded verified inputs from:", candidate.resolve())
        break
if embedded is None:
    raise FileNotFoundError("Save this notebook in Jupyter's current folder, or set NOTEBOOK_FILE to its full path.")
context, history = embedded["context"], embedded["history"]
(CODEDIR/"diagnostic-context.json").write_text(json.dumps(context, indent=2)+"\n", encoding="utf-8")
(OUTPUT/"input-history-check.json").write_text(json.dumps(history, indent=2)+"\n", encoding="utf-8")
print("Original monthly product SHA-256:", context["productSha256"])
print("History check source SHA-256:", embedded["historyFileSha256"])
print("Month:", context["period"]["start"], "to", context["period"]["endExclusive"], "exclusive · UTC")


## 3 · Complete, readable diagnostic Python

The next two cells contain the complete `.py` module and its tests. They export them to the `source` folder for inspection. The original collector and monthly analysis notebook are retained unchanged.

Claims are read from the pocketdex session catalog: a roster member that was paid, or settled for zero, is one settled claim. The boundary query retrieves all blocks in short UTC brackets and sorts their heights numerically in Python. The check then counts:

1. Indexed claims for **sessions ending within the UTC month**, settled at or before the original cutoff: one monthly total per group by session start day (`getNetworkSessionsJson`), corrected at both month edges to session end heights with the sessions of the hour before each edge (`getSessionRostersJson`). The total's highest settlement height must not pass the cutoff.
2. Indexed claims for the **exact original 120 sampled session IDs** in that group, at the same cutoff, in one batch of at most 200 IDs.

The monthly activity count includes unsampled sessions. It uses session **end** heights; the website labels observations by their sampled time. All of the original samples' session ends are checked to lie inside the monthly range. These counts are claim records; they are not earnings totals, unique session counts or all RPC traffic.

New indexed claims at the original cutoff could indicate an indexer backfill or earlier retrieval discrepancy; count agreement alone does not prove every payout amount is identical.


In [ ]:
diagnostic_source = r'''
"""Read-only checks aligned with the saved Pocket fleet analyser month.

Counts are indexed claim records, not requests, distinct sessions or earnings.
No payout amounts, eligibility, parsing, coverage denominators or rankings change.
Claims are read from the pocketdex session catalog: a roster member in a settled
state (paid, or settled for zero) is one settled claim. Monthly counts come from
one getNetworkSessionsJson total by session START day, corrected at both month
edges to the session END height window with getSessionRostersJson.
"""
import hashlib
import json
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

VERSION = "monthly-claims-check-2.0"
GRAPHQL = "https://data.pocket.network/graphql"
EXPECTED_PRODUCT_SHA256 = "feeebfbb120044a782dd60f1879c0613924cb259b7eb528d5f74659055842088"
ROSTERS_QUERY = """query($a:Datetime,$b:Datetime,$apps:[String],$services:[String],$ids:[String],$h:BigInt,$sid:String){
  getSessionRostersJson(rangeStart:$a,rangeEnd:$b,applications:$apps,services:$services,sessionIds:$ids,
  rowLimit:200,beforeHeight:$h,beforeSessionId:$sid)}"""
NETWORK_QUERY = """query($a:Datetime,$b:Datetime,$apps:[String],$services:[String]){
  getNetworkSessionsJson(rangeStart:$a,rangeEnd:$b,applications:$apps,services:$services,byService:true,byApplication:true)}"""
PAGE_ROWS = 200  # Keeps each response well inside the 1 MB limit.
SETTLED_STATES = {"paid", "settled_zero"}


class ProbeError(RuntimeError):
    pass


def utc(value):
    dt = datetime.fromisoformat(value.replace("Z", "+00:00"))
    # Indexer block timestamps sometimes omit the UTC suffix.
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def iso(dt):
    return dt.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def integer(value, label):
    if isinstance(value, bool) or not re.fullmatch(r"\d+", str(value)):
        raise ProbeError(f"Invalid non-negative integer: {label}")
    return int(value)


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, indent=2)+"\n", encoding="utf-8")
    temp.replace(path)


def validate_context(context):
    if context.get("productSha256") != EXPECTED_PRODUCT_SHA256:
        raise ProbeError("This diagnostic is for the exact previously collected month")
    if utc(context["period"]["endExclusive"]) - utc(context["period"]["start"]) != timedelta(days=30):
        raise ProbeError("Wrong month length")
    groups = context["groups"]
    keys = [(r["serviceId"], r["applicationId"]) for r in groups]
    if len(groups) != 34 or len(set(keys)) != 34:
        raise ProbeError("Expected the 34 distinct saved Application/service groups")
    if sum(g["sampledSessions"] for g in groups) != 4080:
        raise ProbeError("Original sampled session denominator differs")
    if sum(g["originalIndexedClaims"] for g in groups) != 47880:
        raise ProbeError("Original sampled claim total differs")
    if sum(g["originalCoveredSessions"] for g in groups) != 1652:
        raise ProbeError("Original covered session total differs")
    if sum(g["originalCoveredSessions"] == 0 for g in groups) != 19:
        raise ProbeError("Original never-covered group count differs")
    all_ids = []
    for g in groups:
        ids = g["sampledSessionIds"]
        if len(ids) != g["sampledSessions"] or len(ids) != len(set(ids)):
            raise ProbeError("Repeated or missing saved session IDs")
        if not all(re.fullmatch(r"[0-9a-f]{64}", sid) for sid in ids):
            raise ProbeError("Unexpected saved session ID format")
        all_ids.extend(ids)
    if len(all_ids) != len(set(all_ids)):
        raise ProbeError("Session ID repeated across groups")
    return {"groups": 34, "sampledSessions": 4080, "originalCoveredSessions": 1652,
            "originalUnknownSessions": 2428, "originalIndexedClaims": 47880}


class Client:
    """Sequential public requests, one attempt each, with bounded responses."""
    def __init__(self, transport=None, timeout=45, progress=print):
        if not 1 <= timeout <= 45:
            raise ValueError("timeout must be between 1 and 45 seconds")
        self.transport = transport or urlopen
        self.timeout = timeout
        self.progress = progress
        self.log = []

    def gql(self, query, variables=None, label="request"):
        self.progress(f"Request {len(self.log)+1}: {label}…")
        body = json.dumps({"query": query, "variables": variables or {}}).encode()
        req = Request(GRAPHQL, data=body,
                      headers={"Content-Type": "application/json", "User-Agent": "POKT-monthly-claims-check/1.0"})
        record = {"label": label, "startedAt": iso(datetime.now(timezone.utc)),
                  "query": query, "variables": variables or {}}
        self.log.append(record)
        began = time.monotonic()
        try:
            with self.transport(req, timeout=self.timeout) as response:
                record["httpStatus"] = getattr(response, "status", 200)
                raw = response.read(1_000_001)
            record["responseBytes"] = len(raw)
            if len(raw) > 1_000_000:
                raise ProbeError("Response exceeds the 1 MB diagnostic limit")
            payload = json.loads(raw)
            if not isinstance(payload, dict):
                raise ProbeError("Response is not a GraphQL object")
            if payload.get("errors"):
                record["graphqlErrors"] = payload["errors"]
                raise ProbeError("GraphQL errors: "+json.dumps(payload["errors"])[:1500])
            if not isinstance(payload.get("data"), dict):
                raise ProbeError("Missing GraphQL data object")
            record["status"] = "succeeded"
            record["data"] = payload["data"]
            return payload["data"]
        except HTTPError as e:
            record["httpStatus"] = e.code
            record["retryAfter"] = e.headers.get("Retry-After", "not supplied")
            raise ProbeError(f"HTTP {e.code}; Retry-After: {record['retryAfter']}. Stop and save the error file.") from e
        except (URLError, TimeoutError, OSError) as e:
            raise ProbeError(f"Connection/timeout failure: {e}. This does not establish an HTTP rate limit.") from e
        except (ValueError, UnicodeError) as e:
            raise ProbeError("Response is not valid GraphQL JSON") from e
        finally:
            record["seconds"] = round(time.monotonic()-began, 3)
            if "status" not in record:
                record["status"] = "failed"


def check_indexer(client, cutoff):
    first = client.gql("query {_metadata {lastProcessedHeight chain}}", label="indexer head")
    metadata = first.get("_metadata") or {}
    if metadata.get("chain") != "pocket":
        raise ProbeError("Indexer chain differs from the saved Pocket product")
    if integer(metadata.get("lastProcessedHeight"), "indexer head") < cutoff:
        raise ProbeError("Indexer is behind the saved cutoff")
    return metadata


def resolve_boundaries(client, context):
    period = context["period"]
    # Retrieve ALL blocks in each ten-minute UTC bracket and choose the lowest
    # numeric height locally. No event-ID or block-ID string ordering is used.
    brackets = [("start", utc(period["start"])), ("end", utc(period["endExclusive"]))]
    clauses = []
    for alias, at in brackets:
        filt = {"timestamp": {"greaterThanOrEqualTo": iso(at), "lessThan": iso(at+timedelta(minutes=10))}}
        # GraphQL input syntax for the known timestamp fields, with quoted values.
        ftext = '{timestamp:{greaterThanOrEqualTo:'+json.dumps(filt["timestamp"]["greaterThanOrEqualTo"])+',lessThan:'+json.dumps(filt["timestamp"]["lessThan"])+'}}'
        clauses.append(f'{alias}:blocks(first:100,filter:{ftext}){{totalCount pageInfo{{hasNextPage}} nodes{{id timestamp}}}}')
    data = client.gql("query {"+" ".join(clauses)+"}", label="exact UTC month boundary blocks")
    result = {}
    for alias, at in brackets:
        conn = data.get(alias) or {}
        rows = conn.get("nodes", [])
        count = integer(conn.get("totalCount"), "boundary block count")
        if not rows or count != len(rows) or conn.get("pageInfo", {}).get("hasNextPage") is not False:
            raise ProbeError("Boundary bracket is empty or truncated; refuse an approximate month")
        heights = [integer(r["id"], "boundary height") for r in rows]
        if len(set(heights)) != len(rows):
            raise ProbeError("Duplicate boundary block")
        rows = sorted(rows, key=lambda r: int(r["id"]))
        if any(not at <= utc(r["timestamp"]) < at+timedelta(minutes=10) for r in rows):
            raise ProbeError("Boundary timestamp outside the requested UTC bracket")
        if any(utc(a["timestamp"]) > utc(b["timestamp"]) for a, b in zip(rows, rows[1:])):
            raise ProbeError("Block height and timestamp order disagree")
        if any(int(b["id"]) != int(a["id"])+1 for a, b in zip(rows, rows[1:])):
            raise ProbeError("Missing indexed block inside boundary bracket")
        result[alias] = {"height": int(rows[0]["id"]), "timestamp": rows[0]["timestamp"],
                         "requestedUTC": iso(at), "bracketBlocks": count}
    lower, upper = result["start"]["height"], result["end"]["height"]
    if not lower < upper <= context["cutoff"]:
        raise ProbeError("Resolved month is beyond the saved settlement cutoff")
    for g in context["groups"]:
        if not lower <= g["minSampledSessionEndHeight"] <= g["maxSampledSessionEndHeight"] < upper:
            raise ProbeError("Saved sampled session ends do not fall inside the resolved month")
    return result


def session_rows(client, label, start=None, end=None, applications=None, services=None, ids=None):
    """Catalog rows, newest first, every page; a range answer must cover it with no settlement gap."""
    rows, before = [], (None, None)
    for page in range(1000):
        variables = {"a": start, "b": end, "apps": applications, "services": services, "ids": ids,
                     "h": None if before[0] is None else str(before[0]), "sid": before[1]}
        result = client.gql(ROSTERS_QUERY, variables, label=f"{label} · page {page+1}").get("getSessionRostersJson") or {}
        part = result.get("data")
        if not isinstance(part, list) or len(part) > PAGE_ROWS:
            raise ProbeError("Invalid session catalog page")
        if ids is None:
            cover = result.get("range") or {}
            if (cover.get("gaps") or not cover.get("covered_from") or not cover.get("covered_to")
                    or utc(cover["covered_from"]) > utc(start) or utc(cover["covered_to"]) < utc(end)):
                raise ProbeError("Session catalog does not cover the month or reports a settlement gap")
        keys = [(integer(r["start_height"], "session start"), r["session_id"]) for r in part]
        if any(b >= a for a, b in zip(keys, keys[1:])) or (before[0] is not None and keys and keys[0] >= before):
            raise ProbeError("Unordered or repeated session catalog page")
        rows.extend(part)
        if ids is not None or len(part) < PAGE_ROWS:
            return rows
        before = keys[-1]
    raise ProbeError("Session catalog pagination did not finish")


def settled_claims(rows, group, cutoff):
    """One claim per member in a settled state, in sessions settled by the saved cutoff."""
    claims = []
    for r in rows:
        if r["application_id"] != group["applicationId"] or r["service_id"] != group["serviceId"]:
            raise ProbeError("Catalog session belongs to another Application/service")
        if r["money_missing"] or "unresolved" in r["member_states"]:
            raise ProbeError("Session money is not indexed yet; do not interpret this result")
        # Sessions longer than the one-hour start margin could end inside the month unseen.
        if integer(r["end_height"], "session end")-integer(r["start_height"], "session start") >= 60:
            raise ProbeError("Session longer than the catalog start margin")
        if r["phase"] != "settled" or integer(r["settled_height"], "settlement block") > cutoff:
            continue
        claims.extend({"id": f"{r['session_id']}:{supplier}", "applicationId": r["application_id"], "serviceId": r["service_id"],
                       "sessionId": r["session_id"], "sessionEndHeight": r["end_height"], "blockId": r["settled_height"]}
                      for supplier, state in zip(r["member_ids"], r["member_states"]) if state in SETTLED_STATES)
    return claims


def monthly_counts(client, context, bounds):
    """Settled claims per (service, Application) for sessions ending inside the month."""
    groups = {(g["serviceId"], g["applicationId"]): g for g in context["groups"]}
    apps = sorted({g["applicationId"] for g in groups.values()})
    services = sorted({g["serviceId"] for g in groups.values()})
    period = context["period"]
    result = client.gql(NETWORK_QUERY, {"a": period["start"], "b": period["endExclusive"], "apps": apps, "services": services},
                        label="monthly settled claims by Application").get("getNetworkSessionsJson") or {}
    cover = result.get("range") or {}
    if (cover.get("gaps") or not cover.get("covered_from") or utc(cover["covered_from"]) > utc(period["start"])
            or utc(cover["covered_to"]) < utc(period["endExclusive"])):
        raise ProbeError("Session catalog does not cover the month or reports a settlement gap")
    counts = {key: 0 for key in groups}
    for row in result.get("data") or []:
        key = (row["service_id"], row["application_id"])
        if key not in counts or utc(row["bucket_start"]) != utc(period["start"]):
            raise ProbeError("Monthly total belongs to another group or bucket")
        if integer(row["active"], "active sessions") or integer(row["unresolved"], "unresolved seats"):
            raise ProbeError("Monthly sessions are not all settled and indexed")
        # The settled heights of the row's sessions: none may be past the saved cutoff.
        if integer(row["last_height"], "last settlement") > context["cutoff"]:
            raise ProbeError("A monthly session settled after the saved cutoff")
        counts[key] += integer(row["paid"], "paid seats")+integer(row["settled_zero"], "settled-zero seats")
    # The total is by session START day; the month is by session END height.
    lower, upper = bounds["start"]["height"], bounds["end"]["height"]
    for edge, at, sign in (("start", period["start"], 1), ("end", period["endExclusive"], -1)):
        rows = session_rows(client, f"month {edge} edge", iso(utc(at)-timedelta(hours=1)), at, apps, services)
        boundary = bounds[edge]["height"]
        for r in rows:
            if integer(r["end_height"], "session end") < boundary:
                continue
            group = groups.get((r["service_id"], r["application_id"]))
            if group is None:
                raise ProbeError("Edge session belongs to an unexpected Application/service")
            counts[(group["serviceId"], group["applicationId"])] += sign*len(settled_claims([r], group, context["cutoff"]))
    if any(n < 0 for n in counts.values()):
        raise ProbeError("Edge correction exceeds a monthly total")
    return {f"{s}|{a}": n for (s, a), n in counts.items()}


def check_group(client, group, cutoff, month_count):
    label = f"{group['serviceId']} · {group['applicationId'][-8:]}"
    sampled_count = 0
    ids = group["sampledSessionIds"]
    for i in range(0, len(ids), 200):
        part = ids[i:i+200]
        found = session_rows(client, f"{label} · sample", ids=part)
        if sorted(r["session_id"] for r in found) != sorted(part):
            raise ProbeError("Saved sampled sessions are missing from the session catalog")
        sampled_count += len(settled_claims(found, group, cutoff))
    if month_count < sampled_count:
        raise ProbeError("Monthly claim total is smaller than its sampled subset; do not interpret this result")
    original = group["originalIndexedClaims"]
    if sampled_count != original:
        finding = "sampled-index-count-changed"
    elif month_count == 0:
        finding = "no-indexed-monthly-claims"
    elif sampled_count == 0:
        finding = "monthly-claims-outside-samples"
    else:
        finding = "sampled-index-count-agrees"
    return {"serviceId": group["serviceId"], "applicationId": group["applicationId"],
            "originalCoveredSessions": group["originalCoveredSessions"], "sampledSessions": group["sampledSessions"],
            "originalSampledClaimCount": original, "recheckedSampledClaimCount": sampled_count,
            "sampledClaimCountDifference": sampled_count-original,
            "monthlyClaimCount": month_count, "finding": finding,
            "interpretation": "Claim-record counts through the saved cutoff. No indexed claim is not verified zero earnings or proof of inactivity."}


def summarise(results):
    findings = {}
    never = [r for r in results if r["originalCoveredSessions"] == 0]
    for row in results:
        findings[row["finding"]] = findings.get(row["finding"], 0)+1
    return {"checkedGroups": len(results), "originalNeverCoveredGroupsChecked": len(never),
            "neverCoveredWithMonthlyClaims": sum(r["monthlyClaimCount"] > 0 for r in never),
            "groupsWithChangedSampledCounts": sum(r["sampledClaimCountDifference"] != 0 for r in results),
            "monthlyClaimRecords": sum(r["monthlyClaimCount"] for r in results),
            "originalSampledClaimRecords": sum(r["originalSampledClaimCount"] for r in results),
            "recheckedSampledClaimRecords": sum(r["recheckedSampledClaimCount"] for r in results),
            "findings": findings}


def validate_checkpoint(state, context, identity):
    if state.get("runIdentity") != identity or state.get("productSha256") != context["productSha256"]:
        raise ProbeError("Existing checkpoint belongs to different evidence; select a new output folder")
    expected = {(r["serviceId"], r["applicationId"]): r for r in context["groups"]}
    seen = set()
    for row in state.get("results", []):
        key = (row["serviceId"], row["applicationId"])
        if key in seen or key not in expected:
            raise ProbeError("Checkpoint contains unexpected or repeated groups")
        seen.add(key)
        group = expected[key]
        if row["originalSampledClaimCount"] != group["originalIndexedClaims"] or row["sampledSessions"] != group["sampledSessions"] or row["originalCoveredSessions"] != group["originalCoveredSessions"]:
            raise ProbeError("Checkpoint original evidence differs")
        month = integer(row["monthlyClaimCount"], "checkpoint monthly claims")
        sampled = integer(row["recheckedSampledClaimCount"], "checkpoint sampled claims")
        if month < sampled or row["sampledClaimCountDifference"] != sampled-group["originalIndexedClaims"]:
            raise ProbeError("Checkpoint claim counts do not reconcile")
    if state.get("status") == "complete":
        if len(seen) != 34 or state.get("summary") != summarise(state["results"]):
            raise ProbeError("Completed checkpoint is incomplete or its summary differs")


def run_probe(context, output, transport=None, progress=print, timeout=45):
    """39 sequential requests on a fresh complete run; checkpoints after groups.

    Indexer head + UTC boundary + one monthly total + one page per month edge,
    then one <=200-ID sample batch per group. No full claim download or
    live-head settlement expansion is done.
    """
    validate_context(context)
    output = Path(output)
    path = output/"monthly-claims-check.json"
    identity = digest({"version": VERSION, "context": context})
    client = Client(transport, timeout, progress)
    if path.exists():
        state = json.loads(path.read_text(encoding="utf-8"))
        validate_checkpoint(state, context, identity)
        if state.get("status") == "complete":
            progress("Completed check already exists. Reusing it with zero new API requests.")
            return state
    else:
        state = {"diagnosticVersion": VERSION, "runIdentity": identity, "productSha256": context["productSha256"],
                 "period": context["period"], "cutoff": context["cutoff"], "status": "running", "results": [],
                 "requestLog": [], "startedAt": iso(datetime.now(timezone.utc)),
                 "monthlyWindowDefinition": "Session end height from first block at/after UTC start to before first block at/after UTC endExclusive. Settlement block <= original cutoff.",
                 "sampleDefinition": "The exact 4,080 session IDs saved in the website's monthly product.",
                 "notice": "Diagnostic counts do not change original payout amounts, denominators, earnings comparisons or ranks."}
    old_logs = list(state.get("requestLog", []))
    try:
        state["indexerMetadata"] = check_indexer(client, context["cutoff"])
        bounds = state.get("boundaryBlocks") or resolve_boundaries(client, context)
        state["boundaryBlocks"] = bounds
        state["monthlyClaimCounts"] = state.get("monthlyClaimCounts") or monthly_counts(client, context, bounds)
        state["status"] = "running"
        state.pop("error", None)
        state["requestLog"] = old_logs+client.log
        atomic_json(path, state)
        done = {(r["serviceId"], r["applicationId"]) for r in state["results"]}
        for group in context["groups"]:
            key = (group["serviceId"], group["applicationId"])
            if key in done:
                continue
            result = check_group(client, group, context["cutoff"], state["monthlyClaimCounts"][f"{key[0]}|{key[1]}"])
            state["results"].append(result)
            state["requestLog"] = old_logs+client.log
            atomic_json(path, state)
            progress(f"Saved {len(state['results'])}/34 groups · {result['finding']}")
        state["status"] = "complete"
        state["completedAt"] = iso(datetime.now(timezone.utc))
        state["summary"] = summarise(state["results"])
        state["requestLog"] = old_logs+client.log
        atomic_json(path, state)
        return state
    except (Exception, KeyboardInterrupt) as e:
        state["status"] = "interrupted" if isinstance(e, KeyboardInterrupt) else "error"
        state["error"] = {"type": type(e).__name__, "message": str(e), "at": iso(datetime.now(timezone.utc))}
        state["requestLog"] = old_logs+client.log
        atomic_json(path, state)
        atomic_json(output/"monthly-claims-error.json", state)
        raise


def compact_diagnostic(state, raw_bytes, snapshot_id):
    """The website's data/monthly-claims-diagnostic.json from one completed check."""
    if state.get("status") != "complete":
        raise ProbeError("Only a completed check can be published")
    log = state["requestLog"]
    statuses = {}
    for record in log:
        key = str(record.get("httpStatus"))
        statuses[key] = statuses.get(key, 0)+1
    return {"diagnosticVersion": state["diagnosticVersion"], "productSha256": state["productSha256"],
            "period": state["period"], "cutoff": state["cutoff"], "completedAt": state["completedAt"],
            "monthlyWindowDefinition": state["monthlyWindowDefinition"], "sampleDefinition": state["sampleDefinition"],
            "notice": state["notice"], "boundaryBlocks": state["boundaryBlocks"], "summary": state["summary"],
            "snapshotId": snapshot_id, "rawDiagnosticSha256": hashlib.sha256(raw_bytes).hexdigest(),
            "indexerMetadata": state["indexerMetadata"], "requestCount": len(log), "httpStatusCounts": statuses,
            "elapsedSeconds": round(sum(r.get("seconds", 0) for r in log), 2),
            "groups": [{k: v for k, v in r.items() if k not in {"exampleMonthlyClaims", "interpretation"}} for r in state["results"]]}
'''
(CODEDIR/"monthly_claims_probe.py").write_text(diagnostic_source, encoding="utf-8", newline="\n")
sys.path.insert(0, str(CODEDIR))
import monthly_claims_probe as probe
probe = importlib.reload(probe)
print("Exported readable Python:", (CODEDIR/"monthly_claims_probe.py").resolve())


In [ ]:
test_source = r'''
import copy
import io
import json
import tempfile
import unittest
from pathlib import Path
from urllib.error import HTTPError, URLError

import monthly_claims_probe as probe


def mock_head(height=960000):
    return {"_metadata": {"chain": "pocket", "lastProcessedHeight": height}}


def mock_boundaries():
    return {"start": {"totalCount": 3, "pageInfo": {"hasNextPage": False},
                      "nodes": [{"id": str(h), "timestamp": f"2026-09-06T00:0{i}:00"}
                                for i, h in reversed(list(enumerate([909000, 909001, 909002])))]},
            "end": {"totalCount": 2, "pageInfo": {"hasNextPage": False},
                    "nodes": [{"id": "953000", "timestamp": "2026-10-06T00:00:00Z"},
                              {"id": "953001", "timestamp": "2026-10-06T00:01:00Z"}]}}


COVERED = {"covered_from": "2026-09-05T00:00:00Z", "covered_to": "2026-10-07T00:00:00Z", "gaps": []}


def session(group, sid, i, claims=0, settled=952200):
    """One catalog row: a session ending inside the month with `claims` paid members."""
    start = 909100+20*i
    members = [f"pokt1member{n}" for n in range(max(claims, 1))]
    return {"session_id": sid, "application_id": group["applicationId"], "service_id": group["serviceId"],
            "start_height": str(start), "end_height": str(start+19), "phase": "settled", "settled_height": str(settled),
            "money_missing": False, "member_ids": members,
            "member_states": ["paid"]*claims if claims else ["idle_no_work"]}


def month_rows(group, extra=0, sample_claims=None):
    """The group's sampled sessions (claims on the first one) plus `extra` unsampled sessions with one claim each."""
    claims = group["originalIndexedClaims"] if sample_claims is None else sample_claims
    rows = [session(group, sid, i, claims if i == 0 else 0) for i, sid in enumerate(group["sampledSessionIds"])]
    rows += [session(group, f"{n:064x}", len(rows)+n, 1) for n in range(extra)]
    return sorted(rows, key=lambda r: (int(r["start_height"]), r["session_id"]), reverse=True)


class ProbeTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.context = json.loads((Path(__file__).parent/"diagnostic-context.json").read_text())

    @staticmethod
    def response(data):
        return io.BytesIO(json.dumps({"data": data}).encode())

    def transport(self, fail_group=None, calls=None, rows=None, monthly=None, network=None):
        """Simulated catalog: monthly totals, two edge pages and the sampled sessions."""
        calls = calls if calls is not None else []
        groups = {(r["serviceId"], r["applicationId"]): r for r in self.context["groups"]}
        first = self.context["groups"][0]
        def send(request, timeout):
            body = json.loads(request.data)
            calls.append(body)
            q, vs = body["query"], body["variables"]
            if "_metadata" in q:
                return self.response(mock_head())
            if "start:blocks" in q:
                return self.response(mock_boundaries())
            if "getNetworkSessionsJson" in q:
                data = network if network is not None else [
                    {"service_id": g["serviceId"], "application_id": g["applicationId"], "bucket_start": "2026-09-06T00:00:00+00:00",
                     "paid": str((monthly or {}).get(k, g["originalIndexedClaims"]+10 if g["originalIndexedClaims"] else 0)),
                     "settled_zero": "0", "active": "0", "unresolved": "0", "first_height": "909100", "last_height": "952300"}
                    for k, g in groups.items() if (monthly or {}).get(k, g["originalIndexedClaims"])]
                return self.response({"getNetworkSessionsJson": {"range": COVERED, "data": data}})
            if vs["ids"] is None:
                # Edge pages: one session crossing each month boundary for the first group, one inside.
                at = vs["b"]
                if at.startswith("2026-09-06"):
                    edge = [session(first, "a"*64, -5, 2)]
                    edge[0].update(start_height="908990", end_height="909009")
                else:
                    edge = [session(first, "b"*64, 0, 1), session(first, "c"*64, 0, 4)]
                    edge[0].update(start_height="952990", end_height="953009")
                    edge[1].update(start_height="952970", end_height="952989")
                return self.response({"getSessionRostersJson": {"range": COVERED, "data": edge}})
            group = next(g for g in groups.values() if vs["ids"][0] in g["sampledSessionIds"])
            key = (group["serviceId"], group["applicationId"])
            if fail_group == key:
                raise URLError("simulated outage")
            data = [r for r in ((rows or {}).get(key) or month_rows(group)) if r["session_id"] in vs["ids"]]
            return self.response({"getSessionRostersJson": {"range": COVERED, "data": data}})
        return send

    def counts(self, client):
        return probe.monthly_counts(client, self.context, probe.resolve_boundaries(client, self.context))

    def test_original_evidence_is_pinned_and_mutation_is_rejected(self):
        self.assertEqual(probe.validate_context(self.context)["originalUnknownSessions"], 2428)
        wrong = copy.deepcopy(self.context)
        wrong["productSha256"] = "wrong"
        with self.assertRaisesRegex(probe.ProbeError, "exact"):
            probe.validate_context(wrong)

    def test_duplicate_session_id_is_rejected(self):
        wrong = copy.deepcopy(self.context)
        wrong["groups"][0]["sampledSessionIds"][1] = wrong["groups"][0]["sampledSessionIds"][0]
        with self.assertRaisesRegex(probe.ProbeError, "Repeated"):
            probe.validate_context(wrong)

    def test_numeric_month_boundaries_are_sorted_locally(self):
        c = probe.Client(lambda *a, **k: self.response(mock_boundaries()), progress=lambda _: None)
        result = probe.resolve_boundaries(c, self.context)
        self.assertEqual(result["start"]["height"], 909000)
        self.assertEqual(result["end"]["height"], 953000)
        self.assertNotIn("orderBy", c.log[0]["query"])

    def test_truncated_or_missing_boundary_is_rejected(self):
        d = mock_boundaries()
        d["start"]["pageInfo"]["hasNextPage"] = True
        c = probe.Client(lambda *a, **k: self.response(d), progress=lambda _: None)
        with self.assertRaisesRegex(probe.ProbeError, "truncated"):
            probe.resolve_boundaries(c, self.context)

    def test_behind_indexer_stops_before_claim_queries(self):
        with self.assertRaisesRegex(probe.ProbeError, "behind"):
            probe.check_indexer(probe.Client(lambda *a, **k: self.response(mock_head(955835)), progress=lambda _: None), 955836)

    def test_exact_month_sample_batches_and_saved_cutoff(self):
        calls = []
        client = probe.Client(self.transport(calls=calls), progress=lambda _: None)
        counts = self.counts(client)
        first = self.context["groups"][0]
        key = f"{first['serviceId']}|{first['applicationId']}"
        # Start edge adds 2 claims (ends at 909009 >= 909000); end edge removes 1 (ends at 953009 >= 953000).
        self.assertEqual(counts[key], first["originalIndexedClaims"]+10+2-1)
        network, start, end = calls[1]["variables"], calls[2]["variables"], calls[3]["variables"]
        self.assertEqual((network["a"], network["b"]), ("2026-09-06T00:00:00Z", "2026-10-06T00:00:00Z"))
        self.assertEqual(len(network["apps"]), len({g["applicationId"] for g in self.context["groups"]}))
        self.assertEqual((start["a"], start["b"], end["a"], end["b"]),
                         ("2026-09-05T23:00:00Z", "2026-09-06T00:00:00Z", "2026-10-05T23:00:00Z", "2026-10-06T00:00:00Z"))
        group = next(g for g in self.context["groups"] if g["sampledSessions"] == 120 and g["originalIndexedClaims"])
        before = json.dumps(group)
        result = probe.check_group(client, group, 955836, counts[f"{group['serviceId']}|{group['applicationId']}"])
        self.assertEqual(len(calls[-1]["variables"]["ids"]), 120)
        self.assertIn("rowLimit:200", calls[-1]["query"])
        self.assertEqual(result["sampledClaimCountDifference"], 0)
        self.assertEqual(json.dumps(group), before)

    def test_monthly_totals_must_be_settled_indexed_and_inside_the_cutoff(self):
        row = {"service_id": self.context["groups"][0]["serviceId"], "application_id": self.context["groups"][0]["applicationId"],
               "bucket_start": "2026-09-06T00:00:00Z", "paid": "1", "settled_zero": "0", "active": "0", "unresolved": "0", "last_height": "952300"}
        for change, message in [({"last_height": "955837"}, "after the saved cutoff"), ({"active": "1"}, "not all settled"),
                                ({"unresolved": "2"}, "not all settled"), ({"application_id": "other"}, "another group")]:
            client = probe.Client(self.transport(network=[{**row, **change}]), progress=lambda _: None)
            with self.assertRaisesRegex(probe.ProbeError, message):
                self.counts(client)

    def test_out_of_month_late_settled_and_foreign_sessions_are_handled(self):
        group = next(g for g in self.context["groups"] if g["originalIndexedClaims"])
        rows = month_rows(group)
        late = session(group, "e"*64, 0, 1, settled=955837)
        early = session(group, "d"*64, -8, 1)
        claims = probe.settled_claims(rows+[late, early], group, 955836)
        self.assertEqual(len(claims), group["originalIndexedClaims"]+1)  # early counted here, filtered by month later
        foreign = copy.deepcopy(rows[0]); foreign["application_id"] = "different"
        with self.assertRaisesRegex(probe.ProbeError, "another Application"):
            probe.settled_claims([foreign], group, 955836)
        missing = copy.deepcopy(rows[0]); missing["money_missing"] = True
        with self.assertRaisesRegex(probe.ProbeError, "not indexed"):
            probe.settled_claims([missing], group, 955836)

    def test_catalog_gaps_and_unordered_pages_are_rejected(self):
        group = self.context["groups"][0]
        def answer(cover, data):
            return lambda *a, **k: self.response({"getSessionRostersJson": {"range": cover, "data": data}})
        client = probe.Client(answer({**COVERED, "gaps": [{"from": "2026-09-10T00:00:00Z", "to": "2026-09-10T01:00:00Z"}]}, []), progress=lambda _: None)
        with self.assertRaisesRegex(probe.ProbeError, "settlement gap"):
            probe.session_rows(client, "x", "2026-09-05T23:00:00Z", "2026-10-06T02:00:00Z", [group["applicationId"]], [group["serviceId"]])
        rows = month_rows(group)
        client = probe.Client(answer(COVERED, list(reversed(rows))), progress=lambda _: None)
        with self.assertRaisesRegex(probe.ProbeError, "Unordered"):
            probe.session_rows(client, "x", "2026-09-05T23:00:00Z", "2026-10-06T02:00:00Z", [group["applicationId"]], [group["serviceId"]])

    def test_no_claims_remain_unknown_instead_of_verified_zero(self):
        group = next(g for g in self.context["groups"] if not g["originalIndexedClaims"])
        result = probe.check_group(probe.Client(self.transport(), progress=lambda _: None), group, 955836, 0)
        self.assertEqual(result["finding"], "no-indexed-monthly-claims")
        self.assertIn("not verified zero", result["interpretation"])
        self.assertNotIn("earnings", result)

    def test_monthly_activity_outside_samples_gets_distinct_finding(self):
        group = next(g for g in self.context["groups"] if not g["originalIndexedClaims"])
        result = probe.check_group(probe.Client(self.transport(), progress=lambda _: None), group, 955836, 7)
        self.assertEqual(result["finding"], "monthly-claims-outside-samples")
        self.assertEqual(result["monthlyClaimCount"], 7)

    def test_new_sampled_claims_are_reported_without_repairing_payouts(self):
        group = next(g for g in self.context["groups"] if not g["originalIndexedClaims"])
        rows = {(group["serviceId"], group["applicationId"]): month_rows(group, sample_claims=3)}
        result = probe.check_group(probe.Client(self.transport(rows=rows), progress=lambda _: None), group, 955836, 9)
        self.assertEqual(result["finding"], "sampled-index-count-changed")
        self.assertEqual(result["sampledClaimCountDifference"], 3)
        with self.assertRaisesRegex(probe.ProbeError, "smaller than its sampled subset"):
            probe.check_group(probe.Client(self.transport(rows=rows), progress=lambda _: None), group, 955836, 2)

    def test_full_probe_is_39_requests_and_complete_cache_uses_none(self):
        calls = []
        with tempfile.TemporaryDirectory() as td:
            before = json.dumps(self.context)
            result = probe.run_probe(self.context, td, self.transport(calls=calls), progress=lambda _: None)
            self.assertEqual(len(calls), 39)  # head, boundaries, monthly total, two edges, 34 sample batches
            self.assertEqual(result["status"], "complete")
            self.assertEqual(result["summary"]["groupsWithChangedSampledCounts"], 0)
            self.assertEqual(result["summary"]["recheckedSampledClaimRecords"], 47880)
            probe.run_probe(self.context, td, self.transport(calls=calls), progress=lambda _: None)
            self.assertEqual(len(calls), 39)
            raw = (Path(td)/"monthly-claims-check.json").read_bytes()
            compact = probe.compact_diagnostic(result, raw, "live-test")
            self.assertEqual((compact["requestCount"], compact["httpStatusCounts"], compact["snapshotId"]), (39, {"200": 39}, "live-test"))
            self.assertNotIn("exampleMonthlyClaims", compact["groups"][0])
            self.assertEqual(json.dumps(self.context), before)

    def test_interruption_saves_error_and_resumes_completed_groups(self):
        second = self.context["groups"][1]
        key = (second["serviceId"], second["applicationId"])
        with tempfile.TemporaryDirectory() as td:
            with self.assertRaisesRegex(probe.ProbeError, "not establish an HTTP rate limit"):
                probe.run_probe(self.context, td, self.transport(fail_group=key), progress=lambda _: None)
            saved = json.loads((Path(td)/"monthly-claims-check.json").read_text())
            self.assertEqual(len(saved["results"]), 1)
            self.assertTrue((Path(td)/"monthly-claims-error.json").exists())
            calls = []
            result = probe.run_probe(self.context, td, self.transport(calls=calls), progress=lambda _: None)
            self.assertEqual(result["status"], "complete")
            self.assertEqual(len(calls), 34)  # indexer head, then 33 remaining sample batches

    def test_incomplete_complete_checkpoint_is_rejected(self):
        with tempfile.TemporaryDirectory() as td:
            probe.run_probe(self.context, td, self.transport(), progress=lambda _: None)
            path = Path(td)/"monthly-claims-check.json"
            d = json.loads(path.read_text())
            d["results"].pop()
            path.write_text(json.dumps(d))
            with self.assertRaisesRegex(probe.ProbeError, "incomplete"):
                probe.run_probe(self.context, td, self.transport(), progress=lambda _: None)

    def test_http_rate_limit_graphql_error_and_invalid_json_are_distinct(self):
        def limited(*a, **k):
            raise HTTPError(probe.GRAPHQL, 429, "rate limit", {"Retry-After": "60"}, None)
        with self.assertRaisesRegex(probe.ProbeError, "HTTP 429; Retry-After: 60"):
            probe.Client(limited, progress=lambda _: None).gql("query{}")
        with self.assertRaisesRegex(probe.ProbeError, "GraphQL errors"):
            probe.Client(lambda *a, **k: io.BytesIO(b'{"errors":[{"message":"schema mismatch"}]}'), progress=lambda _: None).gql("query{}")
        with self.assertRaisesRegex(probe.ProbeError, "not valid GraphQL JSON"):
            probe.Client(lambda *a, **k: io.BytesIO(b'<html>bad</html>'), progress=lambda _: None).gql("query{}")


if __name__ == "__main__":
    unittest.main()
'''
(CODEDIR/"test_monthly_claims_probe.py").write_text(test_source, encoding="utf-8", newline="\n")


In [ ]:
began = time.monotonic()
checks = probe.validate_context(context)
targets = {(g["serviceId"], g["applicationId"]) for g in context["groups"] if not g["originalCoveredSessions"]}
returned = {(g["serviceId"], g["applicationId"]) for g in history["results"]}
assert len(history["results"]) == 19 and targets == returned
assert history["cutoff"] == context["cutoff"]
tests_run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(CODEDIR.resolve()),
                           "-p", "test_monthly_claims_probe.py", "-v"],
                          cwd=CODEDIR.resolve(), capture_output=True, text=True, timeout=60)
print(tests_run.stdout+tests_run.stderr)
if tests_run.returncode:
    raise RuntimeError("Offline diagnostic checks failed. Share this cell before running verify mode.")
print("PASS:", checks)
print(f"Offline checks took {time.monotonic()-began:.1f}s. Tests use simulated responses, with zero real API requests.")


In [ ]:
positive = [r for r in history["results"] if r["indexedHistoryCountThroughCutoff"] > 0]
absent = [r for r in history["results"] if r["indexedHistoryCountThroughCutoff"] == 0]
print(f"{len(positive)} groups with indexed history; {len(absent)} with none through the original cutoff.")
print("72 returned examples predate the month; ID ordering does not establish true latest activity.")
print("Service      Application (last 10 chars)     Historical claim count")
for r in history["results"]:
    print(f"{r['serviceId']:<12} {r['applicationId'][-10:]:<32} {r['indexedHistoryCountThroughCutoff']:>9,}")
print("Historical counts span earlier periods; do not add them to monthly earnings or monthly claims.")


## 4 · Run the small live follow-up on your machine

`MODE = "offline"` skips this step. To run it, change the setting at the top to `"verify"` and choose **Run → Run All Cells**. Let the progress messages finish, then save with **Ctrl+S**.

The result is checkpointed after every group. **If interrupted, rerun without deleting the working folder.** HTTP 429, 403, network timeouts, GraphQL errors, settlement gaps and indexers behind the cutoff are reported distinctly. There is no automatic retry loop. An interrupted run rechecks the indexer head and then continues the remaining groups. A completed run reuses the saved result; choose a new `OUTPUT` folder only if you deliberately want a fresh recheck.

Live access to Pocket's API is unavailable in the authoring environment. The live result below must come from your local run; the supplied 19-group history check is real, and the new live check has not yet been performed here.


In [ ]:
live_result = None
if MODE == "offline":
    print('Live follow-up skipped. Offline checks passed. Set MODE = "verify" at the top, then run all cells.')
else:
    try:
        live_result = probe.run_probe(context, OUTPUT, timeout=REQUEST_TIMEOUT_SECONDS)
        print("PASS: monthly claim-count follow-up completed.")
        print(json.dumps(live_result["summary"], indent=2))
    except (Exception, KeyboardInterrupt) as error:
        print("STOP:", type(error).__name__, str(error))
        print("Saved diagnostic:", (OUTPUT/"monthly-claims-check.json").resolve())
        print("If present, return:", (OUTPUT/"monthly-claims-error.json").resolve())
        print("Save this notebook with Ctrl+S and return it too. Do not start a larger collection to work around this error.")
        raise


## 5 · How the result determines the next work

| Finding | What it means | Next action |
|---|---|---|
| `no-indexed-monthly-claims` | No settled claim records matched the entire monthly session-end range at the saved cutoff. | Retain unknown payouts. Consider an explicitly labelled activity filter for earnings analysis while retaining all eligible Applications for selection analysis. Confirm the intended product definition before adopting it. |
| `monthly-claims-outside-samples` | Indexed monthly activity exists, but none appears in the original sampled session IDs. | Collect a denser seven-day pilot with the same eight services and all eligible Applications, then measure whether it adds useful evidence. |
| `sampled-index-count-changed` | Today's indexer returned a different count for the original sampled IDs at the same cutoff. | Investigate backfilled/revised indexing or previous retrieval discrepancies. Re-fetch and validate affected sampled payouts before updating the product. |
| `sampled-index-count-agrees` | The original sampled claim-record count agrees with the current indexer's count at the saved cutoff. | Retain the existing evidence; review the residual unknown samples separately. Count agreement is a diagnostic, not a payout-amount audit. |

The 34-group control set prevents an analysis based only on the groups with no observed claims. No absent claim is converted into zero earnings. No group is automatically removed from a denominator, and no ranking is unlocked by this notebook.

**After we review this result:** choose the appropriate data improvement, validate the updated product, and check desktop/mobile search, metric sorting, charts, explanations and load performance. Then use the existing source ZIP and detailed Windows guide to create the public GitHub repository and deploy it to Vercel. Refresh that ZIP if the source has changed before upload. Prepare Ramiro's final collection/analysis package after the product and data choices are settled.


In [ ]:
if live_result is not None:
    import csv
    rows = [{k:v for k,v in row.items() if k not in {"exampleMonthlyClaims", "interpretation"}}
            for row in live_result["results"]]
    csv_path = OUTPUT/"monthly-claims-groups.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader(); writer.writerows(rows)
    print("RETURN THIS JSON:", (OUTPUT/"monthly-claims-check.json").resolve())
    print("Optional table:", csv_path.resolve())
else:
    print("OFFLINE PASS. Next: change MODE to verify and run all cells again.")
print("Save this notebook with Ctrl+S. Return the saved notebook with the JSON result or error file.")
print("Original collection and analysis files remain in their existing folders.")
